In [2]:
import os
import numpy as np
import pandas as pd

# Paths: notebook is in zavala_electricity_market, data in dataset/IM-3-GO-WEST
# Run from project root (prj_market) or from zavala_electricity_market
_cwd = os.getcwd()
if os.path.basename(_cwd) == "zavala_electricity_market":
    BASE_DIR = os.path.dirname(_cwd)
else:
    BASE_DIR = _cwd
DATA_DIR = os.path.join(BASE_DIR, "dataset", "IM-3-GO-WEST")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(_cwd, "dataset", "IM-3-GO-WEST")

# Ensure zavala_electricity_market is on path (when running from project root)
import sys
ZAVALA_DIR = os.path.join(BASE_DIR, "zavala_electricity_market")
if os.path.isdir(ZAVALA_DIR) and ZAVALA_DIR not in sys.path:
    sys.path.insert(0, ZAVALA_DIR)

# --- Minimal customization for current repo layout ---

# 1) If the original DATA_DIR doesn't exist, try archive/dataset/IM-3-GO-WEST
if not os.path.isdir(DATA_DIR):
    alt_data_dir = os.path.join(BASE_DIR, "archive", "dataset", "IM-3-GO-WEST")
    if os.path.isdir(alt_data_dir):
        DATA_DIR = alt_data_dir

# 2) If ZAVALA_DIR isn't a valid dir (e.g. code is at repo root),
#    fall back to BASE_DIR itself when it has zavala_funcs.py
if not os.path.isdir(ZAVALA_DIR):
    repo_root_candidate = BASE_DIR
    if os.path.isfile(os.path.join(repo_root_candidate, "zavala_funcs.py")):
        if repo_root_candidate not in sys.path:
            sys.path.insert(0, repo_root_candidate)
        ZAVALA_DIR = repo_root_candidate

print("Using DATA_DIR:", DATA_DIR)
print("Using ZAVALA_DIR:", ZAVALA_DIR)
print("Files:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "not found")

from zavala_funcs import (
    zavala,
    zavala_cvar,
    zavala_deterministic_da,
    zavala_rt_energy_only,
    expected_caps_from_scenarios,
    price_distortion,
    probability_feasible,
    expected_cumulative_regret,
    compute_social_surplus,
    tail_worst_indices_by_value,
    _stack_rt,
)

print("Data dir:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "not found")

Using DATA_DIR: /Users/maxwirattawut/Developer/zavala_electricity_market/dataset/IM-3-GO-WEST
Using ZAVALA_DIR: /Users/maxwirattawut/Developer/zavala_electricity_market
Files: ['nodal_load.csv', 'nodal_wind.csv', 'thermal_gens.csv', 'egrid2023_data_rev2.xlsx', 'nodal_solar.csv']
Data dir: /Users/maxwirattawut/Developer/zavala_electricity_market/dataset/IM-3-GO-WEST
Files: ['nodal_load.csv', 'nodal_wind.csv', 'thermal_gens.csv', 'egrid2023_data_rev2.xlsx', 'nodal_solar.csv']


In [3]:
# Load nodal time series (rows = time, columns = bus_XXXXX)
solar = pd.read_csv(os.path.join(DATA_DIR, "nodal_solar.csv"))
wind = pd.read_csv(os.path.join(DATA_DIR, "nodal_wind.csv"))
load_df = pd.read_csv(os.path.join(DATA_DIR, "nodal_load.csv"))
thermal_df = pd.read_csv(os.path.join(DATA_DIR, "thermal_gens.csv"))

T = len(solar)
assert len(wind) == T and len(load_df) == T, "Solar, wind, load must have same length"
print(f"Time steps: {T}")
print(f"Solar columns: {solar.shape[1]}, Wind: {wind.shape[1]}, Load: {load_df.shape[1]}")
print(f"Thermal generators: {len(thermal_df)}")

Time steps: 8760
Solar columns: 125, Wind: 125, Load: 125
Thermal generators: 280


In [4]:
# Choose buses with non-trivial solar: columns with max > threshold
solar_cols = [c for c in solar.columns if solar[c].max() > 50]
wind_cols = [c for c in wind.columns if wind[c].max() > 50]
# Pick 3 solar and 3 wind (unreliable)
num_solar, num_wind = 3, 3
solar_buses = solar_cols[:num_solar] if len(solar_cols) >= num_solar else list(solar.columns[:num_solar])
wind_buses = wind_cols[:num_wind] if len(wind_cols) >= num_wind else list(wind.columns[:num_wind])

# Reliable: aggregate thermal by bus, pick 4 buses with largest capacity
thermal_by_bus = thermal_df.groupby("Bus")["Max_Cap"].sum().sort_values(ascending=False)
thermal_buses_numeric = list(thermal_by_bus.head(4).index)  # e.g. [408441, 135041, ...]
thermal_bus_cols = [f"bus_{b}" for b in thermal_buses_numeric]  # for load alignment if needed

# Load: use total system load (sum over all buses)
load_total = load_df.sum(axis=1).values  # (T,)

print("Solar buses (unreliable):", solar_buses)
print("Wind buses (unreliable):", wind_buses)
print("Thermal buses (reliable):", thermal_buses_numeric)
print("Load: system total (sum over all buses)")

Solar buses (unreliable): ['bus_100931', 'bus_200261', 'bus_201361']
Wind buses (unreliable): ['bus_100931', 'bus_102281', 'bus_105701']
Thermal buses (reliable): [500991, 605141, 408441, 261001]
Load: system total (sum over all buses)


In [5]:
# Filter thermal_df for the selected buses
selected_buses = [500991, 605141, 408441, 261001]
thermal_selected = thermal_df[thermal_df['Bus'].isin(selected_buses)]
display(thermal_selected)

,Name,Bus,Fuel,Max_Cap,Min_Cap,Heat_Rate
0,PHOENIX_Nuc,408441,NUC (Nuclear),4209.60,1046.90,8.530000
1,TONOPAH_NG,408441,NG (Natural Gas),1325.10,393.07,4.177264
2,PHOENIX_NG,408441,NG (Natural Gas),1207.38,449.72,3.622784
28,BRUSH_C,605141,BIT (Bituminous Coal),552.30,78.79,8.941106
29,WELLINGTON_C,605141,BIT (Bituminous Coal),800.40,247.36,8.242654
30,DENVER_C,605141,BIT (Bituminous Coal),586.31,164.07,7.933221
31,KEENESBURG_NG,605141,NG (Natural Gas),685.11,246.77,5.015047
32,BOULDER_C,605141,BIT (Bituminous Coal),251.00,55.58,8.227582
33,PLATTEVILLE_NG,605141,NG (Natural Gas),1185.48,481.17,2.758167
34,AURORA_NG,605141,NG (Natural Gas),397.80,147.60,4.898308


Debug Logs

In [6]:
# For diagnostics: breakdown of DA and RT allocations
def _log(msg, logfile=None):
    if logfile is None:
        print(msg)
    else:
        with open(logfile, "a", encoding="utf-8") as f:
            f.write(str(msg) + "\n")

def _tech_breakdown_da(g_da):
    g_da = np.asarray(g_da, dtype=float)
    return {
        "solar": g_da[:3].sum(),
        "wind": g_da[3:6].sum(),
        "thermal": g_da[6:10].sum(),
        "total": g_da.sum(),
    }

def _tech_breakdown_rt(G_rt, probs=None):
    G_rt = np.asarray(G_rt, dtype=float)  # shape (S, 10)
    solar = G_rt[:, :3].sum(axis=1)
    wind = G_rt[:, 3:6].sum(axis=1)
    thermal = G_rt[:, 6:10].sum(axis=1)
    total = G_rt.sum(axis=1)

    if probs is None:
        return {
            "solar": solar.mean(),
            "wind": wind.mean(),
            "thermal": thermal.mean(),
            "total": total.mean(),
        }

    probs = np.asarray(probs, dtype=float)
    return {
        "solar": np.dot(probs, solar),
        "wind": np.dot(probs, wind),
        "thermal": np.dot(probs, thermal),
        "total": np.dot(probs, total),
    }

def _print_case_diag(name, probs, g_da, d_da, G_rt, D_rt, pi, Pi, logfile=None):
    g_da = np.asarray(g_da, dtype=float)
    d_da = np.asarray(d_da, dtype=float)
    G_rt = np.asarray(G_rt, dtype=float)
    D_rt = np.asarray(D_rt, dtype=float)

    da = _tech_breakdown_da(g_da)
    rt = _tech_breakdown_rt(G_rt, probs=probs)

    da_load = float(d_da.sum())
    exp_rt_load = float(np.dot(np.asarray(probs, dtype=float), D_rt.sum(axis=1)))

    da_supply = float(g_da.sum())
    exp_rt_supply = float(np.dot(np.asarray(probs, dtype=float), G_rt.sum(axis=1)))

    _log(f"\n===== {name} =====", logfile)
    _log(f"DA price pi: {float(pi):.6f}", logfile)
    _log(f"E[RT price]: {float(np.dot(np.asarray(probs, dtype=float), np.asarray(Pi, dtype=float))):.6f}", logfile)
    _log("DA allocation by tech: " + str({k: round(v, 4) for k, v in da.items()}), logfile)
    _log("Expected RT allocation by tech: " + str({k: round(v, 4) for k, v in rt.items()}), logfile)

    gen_names = [f"solar_{i+1}" for i in range(3)] + [f"wind_{i+1}" for i in range(3)] + [f"thermal_{i+1}" for i in range(4)]
    g_da = np.asarray(g_da, dtype=float)
    exp_rt = np.dot(np.asarray(probs, dtype=float), np.asarray(G_rt, dtype=float))

    df = pd.DataFrame({
        "gen": gen_names,
        "DA": g_da,
        "E_RT": exp_rt,
        "RT_minus_DA": exp_rt - g_da,
    })
    _log(df.to_string(index=False), logfile)
    _log(f"DA load: {da_load:.6f}", logfile)
    _log(f"E[RT load]: {exp_rt_load:.6f}", logfile)
    _log(f"DA supply - DA load: {da_supply - da_load:.6f}", logfile)
    _log(f"E[RT supply] - E[RT load]: {exp_rt_supply - exp_rt_load:.6f}", logfile)

def _print_stoch_vs_cvar_diff(z_g_i, cvar_g_i, logfile=None):
    gen_names = [f"solar_{i+1}" for i in range(3)] + [f"wind_{i+1}" for i in range(3)] + [f"thermal_{i+1}" for i in range(4)]
    z = np.asarray(z_g_i, dtype=float)
    c = np.asarray(cvar_g_i, dtype=float)
    df = pd.DataFrame({
        "gen": gen_names,
        "stoch_DA": z,
        "cvar_DA": c,
        "cvar_minus_stoch": c - z,
    })
    _log("\n===== CVaR - Stochastic DA difference =====", logfile)
    _log(df.to_string(index=False), logfile)
    _log("Tech-level difference: " + str({
        "solar": round((c[:3] - z[:3]).sum(), 4),
        "wind": round((c[3:6] - z[3:6]).sum(), 4),
        "thermal": round((c[6:10] - z[6:10]).sum(), 4),
        "total": round((c - z).sum(), 4),
    }), logfile)

In [7]:
# Initialize log files
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

debug_log = os.path.join(log_dir, "zavala_debug_log.txt")
with open(debug_log, "w", encoding="utf-8") as f:
    f.write("Zavala diagnostics log\n")

Cost Metric

In [8]:
def realized_cost_per_scenario(mc_g_i, g_da, G_rt, d_cap_rt, D_rt, voll=1000.0):
    mc_g_i = np.asarray(mc_g_i, dtype=float)
    g_da = np.asarray(g_da, dtype=float)
    G_rt = np.asarray(G_rt, dtype=float)
    d_cap_rt = np.asarray(d_cap_rt, dtype=float).reshape(-1)
    D_rt = np.asarray(D_rt, dtype=float).reshape(-1)

    mc_delta = mc_g_i / 10.0

    energy_cost = (G_rt * mc_g_i[None, :]).sum(axis=1)
    mismatch_cost = (np.abs(G_rt - g_da[None, :]) * mc_delta[None, :]).sum(axis=1)
    unserved_energy = np.maximum(d_cap_rt - D_rt, 0.0)
    load_shed_cost = voll * unserved_energy

    return energy_cost + mismatch_cost + load_shed_cost

def cvar_from_samples(values, probs, beta=0.95):
    values = np.asarray(values, dtype=float)
    probs = np.asarray(probs, dtype=float)

    order = np.argsort(values)
    v = values[order]
    p = probs[order]
    c = np.cumsum(p)

    var_idx = np.searchsorted(c, beta, side="left")
    var = v[var_idx]

    tail_mask = values >= var
    tail_probs = probs[tail_mask]
    tail_probs = tail_probs / tail_probs.sum()
    cvar = np.dot(tail_probs, values[tail_mask])

    return var, cvar

def risk_adjusted_total_cost(cost_scenarios, probs, beta=0.95):
    E_cost = float(np.dot(probs, cost_scenarios))
    VaR, CVaR = cvar_from_samples(cost_scenarios, probs, beta=beta)
    return {
        "E_cost": E_cost,
        "VaR": VaR,
        "CVaR": CVaR,
        "RiskAdjustedTotalCost": E_cost + CVaR,
    }

Build Real Data

In [9]:
def build_real_data_instance(solar_df, wind_df, load_total_vec, thermal_by_bus, thermal_buses_numeric,
                              solar_buses, wind_buses, start_idx, num_scenarios, rng=None):
    """
    Build (probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar) from real data for one time window.
    - start_idx: first time index
    - num_scenarios: number of consecutive time steps (scenarios)
    """
    if rng is None:
        rng = np.random.default_rng()
    end_idx = start_idx + num_scenarios
    S = num_scenarios

    # Scenario probabilities: near-uniform (same idea as s_real10_mix)
    kappa = 1500.0
    alpha = np.full(S, kappa / S)
    probs = rng.dirichlet(alpha)
    probs = probs / probs.sum()

    # Marginal costs: cheap for unreliable (solar/wind), higher for reliable (thermal)
    mc_unrel = rng.uniform(8.0, 14.0, size=6)
    mc_rel = rng.uniform(35.0, 55.0, size=4)
    mc_g_i = np.concatenate([mc_unrel, mc_rel]).astype(float)

    # Single inelastic load (VOLL)
    mv_d_j = np.array([1000.0], dtype=float)

    # Generator capacities per scenario (S x 10)
    # Columns 0..2: solar, 3..5: wind, 6..9: thermal (constant)
    solar_vals = solar_df.loc[start_idx:end_idx - 1, solar_buses].values  # (S, 3)
    wind_vals = wind_df.loc[start_idx:end_idx - 1, wind_buses].values    # (S, 3)
    unrel_caps = np.clip(np.hstack([solar_vals, wind_vals]), 0.0, None)  # (S, 6)

    rel_caps = np.array([thermal_by_bus[b] for b in thermal_buses_numeric], dtype=float)
    rel_caps = np.broadcast_to(rel_caps, (S, 4))  # (S, 4) constant across scenarios

    g_i_bar = np.hstack([unrel_caps, rel_caps])  # (S, 10)

    # Demand: system total load for each scenario (S x 1)
    d_j_bar = load_total_vec[start_idx:end_idx].reshape(-1, 1).astype(float)
    d_j_bar = np.clip(d_j_bar, 1e-6, None)  # avoid zeros

    return probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar

In [10]:
# Quick sanity check: one small window
NUM_SCENARIOS = 500
rng = np.random.default_rng(42)
probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
    solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
    solar_buses, wind_buses, start_idx=0, num_scenarios=min(NUM_SCENARIOS, T), rng=rng
)
print("probs.shape:", probs.shape, "sum:", probs.sum())
print("g_i_bar.shape:", g_i_bar.shape)
print("d_j_bar.shape:", d_j_bar.shape)
print("Unreliable (solar+wind) sample mean:", g_i_bar[:, :6].mean(axis=0))
print("Reliable (thermal) constant:", g_i_bar[0, 6:])
print("Load sample:", d_j_bar[:5].ravel())

probs.shape: (500,) sum: 1.0
g_i_bar.shape: (500, 10)
d_j_bar.shape: (500, 1)
Unreliable (solar+wind) sample mean: [ 67.33335058  17.28        28.36311076 660.0882676  133.714
 215.59573142]
Reliable (thermal) constant: [6922.33 6837.62 6742.08 6521.69]
Load sample: [75073. 72985. 71226. 70105. 69772.]


In [11]:
def run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar, logfile=None):
    """Run stochastic, CVaR, and deterministic Zavala for one instance. Returns dict of metrics.
    Same logic as run_zavala.py, no changes to external files.
    """
    # ----- Stochastic Zavala -----
    z_g_i, z_d_j, Z_G, Z_D, z_pi, z_Pi = zavala(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    prob_f = probability_feasible(probs, z_g_i, z_d_j, g_i_bar, d_j_bar)
    z_dist = price_distortion(probs, z_pi, z_Pi)
    z_reg = expected_cumulative_regret(probs, z_g_i, z_d_j, z_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_stoch = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=z_g_i, d_da=z_d_j, G_rt=Z_G, D_rt=Z_D)

    # Cost metrics for stochastic case
    stoch_cost_scen = realized_cost_per_scenario(
        mc_g_i, z_g_i, Z_G, d_j_bar[:, 0], Z_D[:, 0], voll=mv_d_j[0]
    )
    stoch_cost_summary = risk_adjusted_total_cost(stoch_cost_scen, probs, beta=0.95)

    # # ----- CVaR Zavala -----
    cvar_g_i, cvar_d_j, C_G, C_D, cvar_pi, cvar_Pi, _ = zavala_cvar(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    cvar_dist = price_distortion(probs, cvar_pi, cvar_Pi)
    cvar_reg = expected_cumulative_regret(probs, cvar_g_i, cvar_d_j, cvar_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_cvar = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=cvar_g_i, d_da=cvar_d_j, G_rt=C_G, D_rt=C_D)

    # Cost metrics for CVaR case
    cvar_cost_scen = realized_cost_per_scenario(
        mc_g_i, cvar_g_i, C_G, d_j_bar[:, 0], C_D[:, 0], voll=mv_d_j[0]
    )
    cvar_cost_summary = risk_adjusted_total_cost(cvar_cost_scen, probs, beta=0.95)

    # ----- Deterministic (expected capacities) -----
    gbar_det, dbar_det = expected_caps_from_scenarios(probs, g_i_bar, d_j_bar)
    g_det, d_det, pi_det = zavala_deterministic_da(mc_g_i, mv_d_j, gbar_det, dbar_det)
    G_det_list, D_det_list, Pi_det_list = [], [], []
    for p in range(len(probs)):
        Gp, Dp, Pi_p = zavala_rt_energy_only(mc_g_i, mv_d_j, g_det, d_det, g_i_bar[p], d_j_bar[p])
        G_det_list.append(Gp)
        D_det_list.append(Dp)
        Pi_det_list.append(Pi_p)
    G_det_rt, D_det_rt = _stack_rt(G_det_list, D_det_list)
    Pi_det = np.array(Pi_det_list)
    det_dist = price_distortion(probs, pi_det, Pi_det)
    det_reg = expected_cumulative_regret(probs, g_det, d_det, pi_det, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_det = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=g_det, d_da=d_det, G_rt=G_det_rt, D_rt=D_det_rt)

    # Cost metrics for deterministic case
    det_cost_scen = realized_cost_per_scenario(
        mc_g_i, g_det, G_det_rt, d_j_bar[:, 0], D_det_rt[:, 0], voll=mv_d_j[0]
    )
    det_cost_summary = risk_adjusted_total_cost(det_cost_scen, probs, beta=0.95)

    # ----- Tail metrics (5% worst by high neg-surplus) -----
    tail = 0.05
    stoch_tail_idx = tail_worst_indices_by_value(ss_stoch["ss_per_scenario"], probs, tail=tail, worst="high")
    cvar_tail_idx = tail_worst_indices_by_value(ss_cvar["ss_per_scenario"], probs, tail=tail, worst="high")
    det_tail_idx = tail_worst_indices_by_value(ss_det["ss_per_scenario"], probs, tail=tail, worst="high")

    stoch_tail_welfare = -np.mean(ss_stoch["ss_per_scenario"][stoch_tail_idx])
    cvar_tail_welfare = -np.mean(ss_cvar["ss_per_scenario"][cvar_tail_idx])
    det_tail_welfare = -np.mean(ss_det["ss_per_scenario"][det_tail_idx])

    stoch_tail_dist = np.mean(np.abs(z_pi - np.array(z_Pi)[stoch_tail_idx]))
    cvar_tail_dist = np.mean(np.abs(cvar_pi - np.array(cvar_Pi)[cvar_tail_idx]))
    det_tail_dist = np.mean(np.abs(pi_det - Pi_det[det_tail_idx]))

    # Log diagnostics/output results analysis
    _print_case_diag("Stochastic", probs, z_g_i, z_d_j, Z_G, Z_D, z_pi, z_Pi, logfile=logfile)
    _print_case_diag("CVaR", probs, cvar_g_i, cvar_d_j, C_G, C_D, cvar_pi, cvar_Pi, logfile=logfile)
    _print_case_diag("Deterministic", probs, g_det, d_det, G_det_rt, D_det_rt, pi_det, Pi_det, logfile=logfile)
    _print_stoch_vs_cvar_diff(z_g_i, cvar_g_i, logfile=logfile)

    _log("\n===== COST SUMMARY =====", logfile)
    _log(
        f"Stochastic | E_cost={stoch_cost_summary['E_cost']:.2f}, "
        f"VaR={stoch_cost_summary['VaR']:.2f}, "
        f"CVaR={stoch_cost_summary['CVaR']:.2f}, "
        f"RiskAdjustedTotalCost={stoch_cost_summary['RiskAdjustedTotalCost']:.2f}",
        logfile
    )
    _log(
        f"CVaR       | E_cost={cvar_cost_summary['E_cost']:.2f}, "
        f"VaR={cvar_cost_summary['VaR']:.2f}, "
        f"CVaR={cvar_cost_summary['CVaR']:.2f}, "
        f"RiskAdjustedTotalCost={cvar_cost_summary['RiskAdjustedTotalCost']:.2f}",
        logfile
    )
    _log(
        f"Deterministic | E_cost={det_cost_summary['E_cost']:.2f}, "
        f"VaR={det_cost_summary['VaR']:.2f}, "
        f"CVaR={det_cost_summary['CVaR']:.2f}, "
        f"RiskAdjustedTotalCost={det_cost_summary['RiskAdjustedTotalCost']:.2f}",
        logfile
    )

    return {
        "prob_feasible": prob_f,
        "stoch_distortion": z_dist, "stoch_regret": z_reg, "stoch_ss": ss_stoch["E_social_surplus"],
        "stoch_tail_welfare": stoch_tail_welfare, "stoch_tail_distortion": stoch_tail_dist,
        "cvar_distortion": cvar_dist, "cvar_regret": cvar_reg, "cvar_ss": ss_cvar["E_social_surplus"],
        "cvar_tail_welfare": cvar_tail_welfare, "cvar_tail_distortion": cvar_tail_dist,
        "det_distortion": det_dist, "det_regret": det_reg, "det_ss": ss_det["E_social_surplus"],
        "det_tail_welfare": det_tail_welfare, "det_tail_distortion": det_tail_dist,
    }

In [12]:
NUM_INSTANCES = 10
NUM_SCENARIOS = 500
rng = np.random.default_rng(2025)

max_start = T - NUM_SCENARIOS
if max_start <= 0:
    raise ValueError(f"Need at least {NUM_SCENARIOS} time steps; have {T}")

# Random start indices for each instance (non-overlapping or random)
start_indices = rng.integers(0, max_start + 1, size=NUM_INSTANCES)

results_list = []
for i in range(NUM_INSTANCES):
    start = int(start_indices[i])
    probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
        solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
        solar_buses, wind_buses, start_idx=start, num_scenarios=NUM_SCENARIOS, rng=rng
    )
    print(f"the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are {probs.shape}, {mc_g_i.shape}, {mv_d_j.shape}, {g_i_bar.shape}, {d_j_bar.shape}")
    res = run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar, logfile=debug_log)
    results_list.append(res)
    print(f"Instance {i+1}/{NUM_INSTANCES} (start={start}) done.")

the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are (500,), (10,), (1,), (500, 10), (500, 1)


/opt/anaconda3/envs/csci2470/lib/python3.11/site-packages/cvxpy/problems/problem.py:167: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "
(CVXPY) Apr 25 08:44:34 PM: Your problem has 5511 variables, 11501 constraints, and 0 parameters.


                                     CVXPY                                     
                                     v1.7.1                                    


(CVXPY) Apr 25 08:44:35 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 25 08:44:35 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 25 08:44:35 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Apr 25 08:44:35 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Apr 25 08:44:36 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Apr 25 08:44:36 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Apr 25 08:44:36 PM: Applying reduction CvxAttr2Constr
(CVXPY) Apr 25 08:44:36 PM: Applying reduction Qp2SymbolicQp


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


...

-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Apr 25 08:54:08 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Apr 25 08:54:12 PM: Applying reduction QpMatrixStuffing
(CVXPY) Apr 25 08:54:40 PM: Applying reduction GUROBI
(CVXPY) Apr 25 08:54:40 PM: Finished problem compilation (took 3.532e+01 seconds).
(CVXPY) Apr 25 08:54:40 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G325)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Non-default parameters:
QCPDual  1

Optimize a model with 56001 rows, 28012 columns and 127511 nonzeros
Model fingerprint: 0x2b992f27
Coefficient statistics:
  Matrix range     [8e-01, 1e+03]
  Objective range  [1e-04, 8e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-01, 1e+05]
Presolve removed 33000 rows and 924 columns
Presolve time: 0.04s
Presolved: 23001 rows, 27088 columns, 88967 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log o

(CVXPY) Apr 25 08:54:42 PM: Problem status: optimal
(CVXPY) Apr 25 08:54:42 PM: Optimal value: -2.930e+07
(CVXPY) Apr 25 08:54:42 PM: Compilation took 3.532e+01 seconds
(CVXPY) Apr 25 08:54:42 PM: Solver (including time spent in interface) took 1.545e+00 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Instance 10/10 (start=8061) done.


In [13]:
# Aggregate and average
keys = list(results_list[0].keys())
means = {k: np.mean([r[k] for r in results_list]) for k in keys}
stds = {k: np.std([r[k] for r in results_list]) for k in keys}

print("============== Real-data results (averaged over {} instances) ================".format(NUM_INSTANCES))
print("Distortion (DA vs E[RT price]):")
print("  Stochastic:", means["stoch_distortion"], "±", stds["stoch_distortion"])
print("  CVaR:", means["cvar_distortion"], "±", stds["cvar_distortion"])
print("  Deterministic:", means["det_distortion"], "±", stds["det_distortion"])
print("E[Social Surplus]:")
print("  Stochastic:", means["stoch_ss"], "±", stds["stoch_ss"])
print("  CVaR:", means["cvar_ss"], "±", stds["cvar_ss"])
print("  Deterministic:", means["det_ss"], "±", stds["det_ss"])
print("Tail (5%) welfare (mean positive SS in worst tail):")
print("  Stochastic:", means["stoch_tail_welfare"], "±", stds["stoch_tail_welfare"])
print("  CVaR:", means["cvar_tail_welfare"], "±", stds["cvar_tail_welfare"])
print("  Deterministic:", means["det_tail_welfare"], "±", stds["det_tail_welfare"])
print("Tail (5%) price distortion:")
print("  Stochastic:", means["stoch_tail_distortion"], "±", stds["stoch_tail_distortion"])
print("  CVaR:", means["cvar_tail_distortion"], "±", stds["cvar_tail_distortion"])
print("  Deterministic:", means["det_tail_distortion"], "±", stds["det_tail_distortion"])
print("Probability feasible:", means["prob_feasible"], "±", stds["prob_feasible"])

============== Real-data results (averaged over 10 instances) ================
Distortion (DA vs E[RT price]):
  Stochastic: 0.16819282522926643 ± 0.07806781490385745
  CVaR: 110.22211520562048 ± 0.08311298545890572
  Deterministic: 327.159378727471 ± 57.63444246622435
E[Social Surplus]:
  Stochastic: 26757738.536068495 ± 132777.23886825083
  CVaR: 26757216.579716254 ± 132566.21914937
  Deterministic: 26451556.706609238 ± 135091.76268224782
Tail (5%) welfare (mean positive SS in worst tail):
  Stochastic: 25806372.615412332 ± 44420.63408768873
  CVaR: 25816493.03541882 ± 43870.25614436774
  Deterministic: 25781098.420765825 ± 42949.94296159389
Tail (5%) price distortion:
  Stochastic: 100.00000000000054 ± 1.038627808283258e-12
  CVaR: 6.109585552904134e-13 ± 2.474392568442086e-13
  Deterministic: 100.0 ± 0.0
Probability feasible: 0.191265120431626 ± 0.11050471512986863


In [14]:
summary = pd.DataFrame({
    "Method": ["Stochastic", "CVaR", "Deterministic"] * 3,
    "Metric": ["Distortion", "Distortion", "Distortion", "E[SS]", "E[SS]", "E[SS]", "Tail welfare", "Tail welfare", "Tail welfare"],
    "Mean": [
        means["stoch_distortion"], means["cvar_distortion"], means["det_distortion"],
        means["stoch_ss"], means["cvar_ss"], means["det_ss"],
        means["stoch_tail_welfare"], means["cvar_tail_welfare"], means["det_tail_welfare"],
    ],
    "Std": [
        stds["stoch_distortion"], stds["cvar_distortion"], stds["det_distortion"],
        stds["stoch_ss"], stds["cvar_ss"], stds["det_ss"],
        stds["stoch_tail_welfare"], stds["cvar_tail_welfare"], stds["det_tail_welfare"],
    ],
})
summary["Std/Mean"] = summary["Std"] / summary["Mean"]
display(summary)

,Method,Metric,Mean,Std,Std/Mean
0,Stochastic,Distortion,1.681928e-01,0.078068,0.464157
1,CVaR,Distortion,1.102221e+02,0.083113,0.000754
2,Deterministic,Distortion,3.271594e+02,57.634442,0.176166
3,Stochastic,E[SS],2.675774e+07,132777.238868,0.004962
4,CVaR,E[SS],2.675722e+07,132566.219149,0.004954
5,Deterministic,E[SS],2.645156e+07,135091.762682,0.005107
6,Stochastic,Tail welfare,2.580637e+07,44420.634088,0.001721
7,CVaR,Tail welfare,2.581649e+07,43870.256144,0.001699
8,Deterministic,Tail welfare,2.578110e+07,42949.942962,0.001666
